# Example 11.3.1.3 / 2: Reproduce Random Variates

Companion notebook for **11.3.1.3 GSL Probability Distributions** in the AMPL lab Documentation, section "Reproduce Random Variates". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Reproduce Random Variates

All random variates share one generator, seeded from the AMPL option `randseed`. The same seed
gives the same sequence, and `gsl_rng_set` chooses a different generator algorithm by name. The
`call` statement has no amplpy method, so the amplpy box sends it with `ampl.eval`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
load amplgsl.dll;
function gsl_ran_gaussian random;  # mean-0 normal variate, new value per call
function gsl_rng_set;              # selects the generator algorithm by name

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.option["randseed"] = 5
first = ampl.get_value("gsl_ran_gaussian(1)")
ampl.option["randseed"] = 5
print("same seed, same draw:", first == ampl.get_value("gsl_ran_gaussian(1)"), round(first, 6))
ampl.eval("call gsl_rng_set('taus2');")
ampl.option["randseed"] = 781206
print("taus2 draw:", round(ampl.get_value("gsl_ran_gaussian(1)"), 6))

same seed, same draw: True 0.099644
taus2 draw: 0.618499


### Pure AMPL: model.run

The same in pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option randseed 5;
display gsl_ran_gaussian(1);
option randseed 5;
display gsl_ran_gaussian(1);
call gsl_rng_set('taus2');
option randseed 781206;
display gsl_ran_gaussian(1);

gsl_ran_gaussian(1) = 0.0996439

gsl_ran_gaussian(1) = 0.0996439

gsl_ran_gaussian(1) = 0.618499

